In [11]:
import pandas as pd
import numpy as np

# =========================
# 0) 데이터 로드 (이미 df가 있으면 이 블록은 생략)
# =========================
df = pd.read_csv("/content/가맹점월_압축데이터.csv")


In [13]:
# 업종별 데이터 개수 (중복 제거)
industry_counts_unique = (
    df.drop_duplicates(subset=['업종', '가맹점구분번호'])
    .groupby('업종')['가맹점구분번호']
    .nunique()
    .reset_index(name='가맹점수')
    .sort_values('가맹점수', ascending=False)
)

# 상권별 데이터 개수 (중복 제거)
trade_counts_unique = (
    df.drop_duplicates(subset=['상권', '가맹점구분번호'])
    .groupby('상권')['가맹점구분번호']
    .nunique()
    .reset_index(name='가맹점수')
    .sort_values('가맹점수', ascending=False)
)

industry_counts_unique, trade_counts_unique


(           업종  가맹점수
 65   한식-육류/고기   442
 52         카페   357
 19     백반/가정식   346
 63  한식-단품요리일반   306
 50        축산물   285
 ..        ...   ...
 24       스테이크     1
 33        유제품     1
 31     와플/크로플     1
 54        탕후루     1
 64    한식-두부요리     1
 
 [73 rows x 2 columns],
             상권  가맹점수
 9           성수   762
 14         왕십리   538
 4           뚝섬   468
 18         한양대   328
 5          마장동   257
 1         금남시장   249
 2          답십리   179
 13          옥수   121
 10         신금호   100
 19          행당    85
 16      장한평자동차    41
 3   동대문역사문화공원역     1
 0         건대입구     1
 12          오남     1
 11      압구정로데오     1
 7          방배역     1
 8          서면역     1
 6        미아사거리     1
 15          자양     1
 17        풍산지구     1
 20        화양시장     1)

In [5]:
# 필요한 컬럼 존재 확인
required_cols = ['업종', '가맹점구분번호', '기준년월', '동일 업종 매출금액 비율']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"다음 컬럼이 누락되어 있습니다: {missing}")

# =========================
# 1) 전처리 및 정렬
# =========================
# 기준년월을 정수형으로 정렬(문자면 astype(int) 가능)
df = df.copy()
df['기준년월'] = df['기준년월'].astype(int)
df = df.sort_values(['업종', '가맹점구분번호', '기준년월'])

# =========================
# 2) 전월 대비 '동일 업종 매출금액 비율'로 성장률 계산
#    성장률 = (금월비율 - 전월비율) / 전월비율
# =========================
df['전월비율'] = df.groupby(['업종', '가맹점구분번호'])['동일 업종 매출금액 비율'].shift(1)

# 0으로 나누기/무한대 처리 방지:
# - 전월비율이 0인 경우:
#     * 금월비율 > 0 : +inf 로 해석될 수 있으므로 NaN으로 처리(하위 10% 산출 왜곡 방지)
#     * 금월비율 = 0 : 0/0 → NaN
with np.errstate(divide='ignore', invalid='ignore'):
    growth = (df['동일 업종 매출금액 비율'] - df['전월비율']) / df['전월비율']

# 전월비율이 0인 경우는 NaN 처리
zero_prev = (df['전월비율'] == 0)
growth = growth.mask(zero_prev, np.nan)

# 무한대/이상치 제거
growth = growth.replace([np.inf, -np.inf], np.nan)

df['동일업종내매출성장률'] = growth

# 성장률 계산 불가(첫 달 등) 행 제거
df_growth = df.dropna(subset=['동일업종내매출성장률']).copy()

# =========================
# 3) 업종별 하위 10% 관측치 추출
# =========================
# 업종별 하위 10% 컷오프(quantile 0.1)
thr = (
    df_growth.groupby('업종', as_index=False)['동일업종내매출성장률']
    .quantile(0.10)
    .rename(columns={'동일업종내매출성장률': '하위10퍼센트기준'})
)

# 컷오프 병합
df_growth = df_growth.merge(thr, on='업종', how='left')

# 업종별 하위 10% 관측치
df_bottom10 = df_growth[df_growth['동일업종내매출성장률'] <= df_growth['하위10퍼센트기준']].copy()

# =========================
# 4) (요청) 업종별 하위 10% '가맹점구분번호'만 뽑기
#    - 업종별로 중복 제거한 가맹점 리스트/셋 반환
# =========================
# (A) 업종별 가맹점 set 사전
industry_to_merchants = (
    df_bottom10.groupby('업종')['가맹점구분번호']
    .apply(lambda s: sorted(set(s)))
    .to_dict()
)

# (B) 확인용 테이블(업종, 가맹점구분번호 - 중복 제거)
bottom10_merchants_by_industry = (
    df_bottom10[['업종', '가맹점구분번호']]
    .drop_duplicates()
    .sort_values(['업종', '가맹점구분번호'])
    .reset_index(drop=True)
)

# =========================
# 5) (선택) 저장
# =========================
# df_bottom10.to_csv("업종별_하위10퍼_관측치.csv", index=False, encoding="utf-8-sig")
bottom10_merchants_by_industry.to_csv("업종별_하위10퍼_가맹점목록.csv", index=False, encoding="utf-8-sig")

# =========================
# 6) 출력 예시
# =========================
print("=== 업종별 하위 10% 가맹점구분번호 (사전 형태) ===")
for k, v in industry_to_merchants.items():
    print(k, ":", v[:15], ("... (+%d more)" % (len(v)-15) if len(v) > 15 else ""))

print("\n=== 표 형태 미리보기 (상위 20행) ===")
print(bottom10_merchants_by_industry.head(20))


=== 업종별 하위 10% 가맹점구분번호 (사전 형태) ===
건강식품 : ['18185E33A8', '18822DD6FB', '4DAE33EA04', '62A9B7888B', 'B4A7FA3204', 'CA4136E334', 'DA3D8B1FB0'] 
건강원 : ['0C52600659', '1112ACEF8D', '2FC6C4F505', '4F4CDEDF32', '62AC8476D3', '643B6CDCC2', 'B94F386154', 'E5E8F3F456'] 
건어물 : ['9676392FCC'] 
구내식당/푸드코트 : ['56CE6776C6', 'C8BFE622A5'] 
기사식당 : ['389E239904'] 
기타세계요리 : ['153D5AC634', '260F71BF9B', '2DD4A5D048', 'CA60093BA7', 'E811B2836B', 'E8BDFFAD07', 'FC4D2DC52C'] 
꼬치구이 : ['07E06697D9', '2FC5FFC0AA', '39A0648E6F', '9350A78BD7', '9FFE31B3A0', 'DE5FF7193C', 'F1B5E625FC', 'F1F48E332E', 'F79C44E620', 'FB709A768C', 'FFF0DAC445'] 
농산물 : ['0AE494A908', '0CF1A5859F', '0F0676C7F0', '178A3B9A58', '1BB00863B9', '348CC434D7', '50FB1F6C45', '5424DBE00D', '5509AAD8EE', '56DDDE972B', '57E5E99129', '685929E817', '6D3DAB1193', '6F49465F62', '7D693A219F'] ... (+17 more)
담배 : ['15EE2AE836', '3181D01F3F', '74D5125FD6', '8234005757', '9748DB8705', 'E3B9A9EE3A'] 
도너츠 : ['39975534D9', '759BF53FF1', 'EB654FE0E6', 'EE4985

In [10]:
import pandas as pd
import numpy as np


# df와 동일업종내매출성장률이 앞 단계에서 계산되어 있다고 가정합니다.
# (안 되어 있다면 아래 "재계산 블록"을 실행합니다.)

needed_cols = ['업종', '상권', '가맹점구분번호', '기준년월', '동일 업종 매출금액 비율']
if not set(needed_cols).issubset(set(df.columns)):
    raise ValueError(f"필요 컬럼 누락: {set(needed_cols) - set(df.columns)}")

if '동일업종내매출성장률' not in df.columns:
    # ===== 재계산 블록 (전월 대비 '동일 업종 매출금액 비율' 성장률) =====
    df = df.copy()
    df['기준년월'] = df['기준년월'].astype(int)
    df = df.sort_values(['업종', '가맹점구분번호', '기준년월'])
    df['전월비율'] = df.groupby(['업종', '가맹점구분번호'])['동일 업종 매출금액 비율'].shift(1)

    with np.errstate(divide='ignore', invalid='ignore'):
        growth = (df['동일 업종 매출금액 비율'] - df['전월비율']) / df['전월비율']

    zero_prev = (df['전월비율'] == 0)
    growth = growth.mask(zero_prev, np.nan).replace([np.inf, -np.inf], np.nan)
    df['동일업종내매출성장률'] = growth

# 성장률 유효치만 사용
df_growth = df.dropna(subset=['동일업종내매출성장률']).copy()

# =========================
# 1) "상권별" 하위 10% 컷 구하기
# =========================
trade_thr = (
    df_growth.groupby('상권', as_index=False)['동일업종내매출성장률']
    .quantile(0.10)
    .rename(columns={'동일업종내매출성장률': '상권_하위10퍼센트기준'})
)

# 병합 후 하위 10% 관측치만 필터
df_growth_trade = df_growth.merge(trade_thr, on='상권', how='left')
df_bottom10_trade = df_growth_trade[
    df_growth_trade['동일업종내매출성장률'] <= df_growth_trade['상권_하위10퍼센트기준']
].copy()

# =========================
# 2) 상권별 하위 10% 가맹점구분번호만 뽑기
# =========================
# (A) 상권별 딕셔너리 (set → list 정렬)
trade_to_merchants = (
    df_bottom10_trade.groupby('상권')['가맹점구분번호']
    .apply(lambda s: sorted(set(s)))
    .to_dict()
)

# (B) 표(중복 제거)
bottom10_merchants_by_trade = (
    df_bottom10_trade[['상권', '가맹점구분번호']]
    .drop_duplicates()
    .sort_values(['상권', '가맹점구분번호'])
    .reset_index(drop=True)
)

# =========================
# 3) 저장 및 미리보기
# =========================
# 관측치와 목록 CSV로 저장
df_bottom10_trade.to_csv("상권별_하위10퍼_관측치.csv", index=False, encoding="utf-8-sig")
bottom10_merchants_by_trade.to_csv("상권별_하위10퍼_가맹점목록.csv", index=False, encoding="utf-8-sig")

# # 미리보기(표를 UI로 보여주기)
# display_dataframe_to_user("상권별 하위10% 가맹점 목록 (중복 제거)", bottom10_merchants_by_trade.head(200))

# 간단 출력을 위해 상위 몇 개만 표기
preview_lines = []
for k in sorted(trade_to_merchants.keys())[:10]:
    merchants = trade_to_merchants[k]
    preview_lines.append(f"{k}: {merchants[:10]}{' ...' if len(merchants) > 10 else ''}")
"\n".join(preview_lines)


"건대입구: ['BBC08E8D52']\n금남시장: ['018DD7E495', '01B82F195A', '0616E5FFD3', '078E739D39', '0840AC0A17', '09B5A19D91', '0C52600659', '0C5EE2823A', '0DABCC3BB8', '11191365FD'] ...\n답십리: ['022FF198C7', '03360C97C5', '090DD25F79', '105CC28499', '162D2613BC', '19D3456A64', '1A185BC72B', '1B554AAC24', '1CB111F32B', '1DF52E7C5F'] ...\n동대문역사문화공원역: ['ACDE665CFC']\n뚝섬: ['000F03E44A', '003473B465', '0080644746', '01EBAA3D0F', '01FD0FCC3D', '04A4FBD34D', '055EDDDD01', '06A87DAA45', '092717B584', '09A5C8DB83'] ...\n마장동: ['01AB37892D', '02826B57BE', '0624A8697D', '07048B6141', '08108678CB', '09D2771DD5', '0DBD011FE1', '0E8F0F8410', '104031A6C2', '1109D8FCC5'] ...\n미아사거리: ['5B4D253AB9']\n방배역: ['70F032FFD5']\n서면역: ['FE3AAF5CCD']\n성수: ['0050D68B18', '00803E9174', '00F733C995', '019E31E4C6', '01F74A3D59', '01FDC9FAEA', '0305234DDB', '036B3FD49B', '03DCE2BB5C', '04F03995B9'] ..."